In [1]:
# Recommender: Last Year High, Recent Sales Low

import pandas as pd
from datetime import datetime

# 1. Load and clean data
prev_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\YTD 2024-2025 NC_E.xlsx")
curr_df = pd.read_excel(r"D:\OneDrive - Nilons Enterprises Pvt Ltd\Desktop\Anadi\Data\SAP25_apr1st_may31st_E.xlsx")

# Standardize column names
prev_df.columns = prev_df.columns.str.strip()
curr_df.columns = curr_df.columns.str.strip()

# Convert Invoice Date to datetime
prev_df['Invoice Date'] = pd.to_datetime(prev_df['Invoice Date'])
curr_df['Invoice Date'] = pd.to_datetime(curr_df['Invoice Date'])

# Ensure consistent naming
prev_df.rename(columns={'Billing Amount': 'Bill Amount', 'Distributor Code': 'Distributor'}, inplace=True)
curr_df.rename(columns={'C. No': 'Distributor', 'C. Name': 'Distributor Name', 'C. Area': 'Area'}, inplace=True)

# Ensure Bill Amount is numeric
curr_df['Bill Amount'] = pd.to_numeric(curr_df['Bill Amount'], errors='coerce')
prev_df['Bill Amount'] = pd.to_numeric(prev_df['Bill Amount'], errors='coerce')

# 2. Set date references
now = pd.to_datetime("2025-05-01")  # set to May 2025
this_month = now.to_period("M").strftime('%Y-%m')
last_month = (now - pd.DateOffset(months=1)).to_period("M").strftime('%Y-%m')
two_months_ago = (now - pd.DateOffset(months=2)).to_period("M").strftime('%Y-%m')
last_year_same_month = (now - pd.DateOffset(years=1)).to_period("M").strftime('%Y-%m')

# 3. Create period columns
for df in [prev_df, curr_df]:
    df['period'] = df['Invoice Date'].dt.to_period("M").astype(str)

# Combine both datasets
data = pd.concat([prev_df, curr_df], ignore_index=True)

# 4. Aggregate sales
sales = data.groupby(['Distributor', 'Item Code', 'period'])['Bill Amount'].sum().reset_index()

# 5. Pivot table for features
sales_pivot = sales.pivot_table(
    index=['Distributor', 'Item Code'],
    columns='period',
    values='Bill Amount',
    fill_value=0
).reset_index()

# 6. Feature engineering with safe fallback
sales_pivot['last_year_same_month_sales'] = (
    sales_pivot[last_year_same_month] if last_year_same_month in sales_pivot.columns else 0
)
sales_pivot['current_month_sales'] = (
    sales_pivot[this_month] if this_month in sales_pivot.columns else 0
)
sales_pivot['avg_last_2_months'] = (
    (sales_pivot[last_month] if last_month in sales_pivot.columns else 0) +
    (sales_pivot[two_months_ago] if two_months_ago in sales_pivot.columns else 0)
) / 2

# 7. Recommendation condition
recommendations = sales_pivot[
    sales_pivot['last_year_same_month_sales'] >
    (sales_pivot['avg_last_2_months'] + sales_pivot['current_month_sales'])
].copy()

# 8. Add Item Name for clarity
item_lookup = data[['Item Code', 'Item Name']].drop_duplicates()
recommendations = recommendations.merge(item_lookup, on='Item Code', how='left')

# 9. Final output
cols_to_show = [
    'Distributor', 'Item Code', 'Item Name',
    'last_year_same_month_sales', 'avg_last_2_months', 'current_month_sales'
]
recommendations = recommendations[cols_to_show].sort_values(
    by=['Distributor', 'last_year_same_month_sales'], ascending=[True, False]
)

# Display sample
recommendations.head(20)

,Distributor,Item Code,Item Name,last_year_same_month_sales,avg_last_2_months,current_month_sales
0,100006,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...,224960.0,76050.0,0.0
1,100006,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI,224960.0,76050.0,0.0
2,100013,1400919.0,15 KG 5-6 MM PAPAYA FRUIT PRESERVED RED POU (K...,421800.0,0.0,0.0
3,100013,1400919.0,15Kg*1 5-6MM CANDIED FRUIT RED POU KRCHI,421800.0,0.0,0.0
22,100015,1400840.0,15 KG POPULAR PAPAYA FRUIT PRESERVED MIX POU,43902.0,7425.0,0.0
23,100015,1400840.0,15Kg*1 POP CANDIED FRUIT MIX POU,43902.0,7425.0,0.0
20,100015,1400838.0,15 KG POPULAR PAPAYA FRUIT PRESERVED RED POU,41190.0,7425.0,0.0
21,100015,1400838.0,15Kg*1 POP CANDIED FRUIT RED POU,41190.0,7425.0,0.0
6,100015,1400022.0,400 GM STD MIX PICKLE BTL M/O,7712.0,0.0,0.0
7,100015,1400022.0,400g*20 STD MIX PICKLE BTL MO,7712.0,0.0,0.0
